In [ ]:
import numpy as np

# Activation Functions
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x>0).astype(float)

def sigmoid(x):
    return 1/(1+np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1-s)

In [ ]:
class CustomNN:
    def __init__(self, hidden_sizes:list, input_size, output_size):
        self.W = []
        self.b = []
        
        prev_hidden_size = input_size
        i = 0
        for hidden_size in hidden_sizes: 
            self.W.append(np.random.randn(hidden_size, prev_hidden_size) * np.sqrt(2. / hidden_size))
            self.b.append(np.random.uniform(-1, 1, (hidden_size, 1)))
            prev_hidden_size = hidden_size
            i += 1
        
        self.W.append(np.random.randn(output_size, prev_hidden_size) * np.sqrt(2. / output_size))
        self.b.append(np.random.uniform(-1, 1, (output_size, 1)))
        pass

    def forward(self, x):
        x = np.array(x).reshape(-1, 1)

        # Hidden Layer
        self.z = []
        for W,b in zip(self.W[:-1],self.b[:-1]):
            self.z.append(np.dot(W, x) + self.b)
            x = relu(self.z[-1])
            self.a.append(x)

        # Output Layer
        self.z_out = np.dot(self.W[-1], self.a[-1]) + self.b[-1]
        self.a_out = sigmoid(self.z_out)

        self.z.append(self.z_out)
        self.a.append(self.a_out)

        return self.a_out
    
    def backward(self, x, y, lr=0.01):
        x = np.array(x).reshape(-1, 1)
        y = np.array(y).reshape(-1, 1)

        # Output Layer error
        dz_out = (self.a[-1] - y) * sigmoid_derivative(self.z[-1])
        dW_out = np.dot(dz_out, self.a[-2].T)
        db_out = dz_out


        # Hidden Layer error
        dz_prev = dz_out
        dW = []
        db = []
        for W,z in zip(self.W[:-1], self.z[:-1]):
            dz = np.dot(W.T, dz_prev) * relu_derivative(z)
            dW.append(np.dot(dz, x.T))
            dz_prev = dz
            db.append(dz)

        # Update weights and biases
        self.W[-1] -= lr * dW_out
        self.b[-1] -= lr * db_out

        for W,b,dW,db in zip(self.W[:-1], self.b[:-1], dW, db):
            W -= lr * dW
            b -= lr * db

    def train(self, X, Y, epochs=1000, lr=0.01):
        for _ in range(epochs):
            for x,y in zip(X, Y):
                self.forward(x)
                self.backward(x, y, lr)
                

class ShallowNN(CustomNN):
    def __init__(self):
        super().__init__(hidden_sizes=[20], input_size=2, output_size=1);

class DeepNN(CustomNN):
    def __init__(self):
        super().__init__(hidden_sizes=[10,10], input_size=2, output_size=1)


In [ ]:
# --- Example Usage --- 
X = [[0,0], [0,1], [1,0], [1,1]] # XOR inputs 
Y = [0, 1, 1, 0] # XOR outputs 
nn = ShallowNN() 
nn.train(X, Y, epochs=5000, lr=0.1) 
for x in X: print(f"Input: {x}, Output: {nn.forward(x)[0][0]:.4f}")